In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import geopandas as gpd

# Locate the project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

candidate_path = (
    project_root
    / "data"
    / "interim"
    / "chinese_restaurant_candidates.csv"
)

all_candidates = pd.read_csv(
    candidate_path,
    dtype="string"
)

status_summary = (
    all_candidates["chinese_candidate_status"]
    .value_counts(dropna=False)
    .rename_axis("chinese_candidate_status")
    .reset_index(name="facility_count")
)

display(status_summary)

# Keep the unresolved but plausible cuisine candidates
review_candidates = all_candidates.loc[
    all_candidates["chinese_candidate_status"]
    == "review_candidate"
].copy()

print("Review candidates:", len(review_candidates))

print(
    "Duplicate facility IDs:",
    review_candidates["facility_id"]
    .duplicated()
    .sum()
)

address_columns = [
    "facility_address",
    "facility_city",
    "facility_state",
    "facility_zip"
]

missing_address_summary = (
    review_candidates[address_columns]
    .isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_count")
)

display(missing_address_summary)

preview_columns = [
    column
    for column in [
        "facility_id",
        "facility_name",
        "facility_address",
        "facility_city",
        "facility_state",
        "facility_zip",
        "matched_review_keywords"
    ]
    if column in review_candidates.columns
]

display(
    review_candidates[preview_columns].head(20)
)

,chinese_candidate_status,facility_count
0,strong_candidate,559
1,review_candidate,270
2,review_likely_other_cuisine,24


Review candidates: 270
Duplicate facility IDs: 0


,column,missing_count
0,facility_address,0
1,facility_city,0
2,facility_state,0
3,facility_zip,0


,facility_id,facility_name,facility_address,facility_city,facility_state,facility_zip,matched_review_keywords
16,FA0013677,FIRST SZECHWAN WOK,10855 LINDBROOK DR,LOS ANGELES,CA,90024,WOK
21,FA0014798,GARDEN WOK,6117 RESEDA BLVD STE B&C,RESEDA,CA,91335,WOK
30,FA0016232,HAI PHONG NOODLES,10990 LOWER AZUSA RD STE 2,EL MONTE,CA,91731,NOODLES
36,FA0017693,HUI TOU XIANG NOODLE HOUSE,704 W LAS TUNAS DR STE 5,SAN GABRIEL,CA,91776,"XIANG, NOODLE"
39,FA0019766,DRAGON HERBS,460 S ROBERTSON BLVD,LOS ANGELES,CA,90048,DRAGON
40,FA0020085,EAGLEROCK GREEN DRAGON,1733 COLORADO BLVD,LOS ANGELES,CA,90041,DRAGON
41,FA0020314,KOTOHIRA NOODLES,1747 W REDONDO BEACH BLVD,GARDENA,CA,90247,NOODLES
44,FA0023057,LUCKY WOK,8527 E ALONDRA BLVD STE 172,PARAMOUNT,CA,90723,WOK
48,FA0026043,GOLDEN NOODLE & GRILL,965 S GLENDORA AVE,WEST COVINA,CA,91790,NOODLE
49,FA0026066,GOLDEN WOK,3403 W SLAUSON AVE,LOS ANGELES,CA,90043,WOK


In [5]:
import re

def clean_address_for_geocoding(address):
    if pd.isna(address):
        return pd.NA

    address = str(address).upper().strip()

    unit_patterns = [
        r"\s+(?:STE|SUITE)\s*#?\s*[A-Z0-9\-&]+.*$",
        r"\s+(?:UNIT|BLDG|BUILDING|ROOM|RM)\s*#?\s*[A-Z0-9\-&]+.*$",
        r"\s+#\s*[A-Z0-9\-&]+.*$"
    ]

    for pattern in unit_patterns:
        address = re.sub(pattern, "", address)

    address = re.sub(r"\s+", " ", address).strip()
    return address


review_geocoding_batch = review_candidates[
    [
        "facility_id",
        "facility_address",
        "facility_city",
        "facility_state",
        "facility_zip"
    ]
].copy()

review_geocoding_batch["original_address"] = (
    review_geocoding_batch["facility_address"]
)

review_geocoding_batch["facility_address"] = (
    review_geocoding_batch["facility_address"]
    .apply(clean_address_for_geocoding)
)

review_geocoding_batch["facility_zip"] = (
    review_geocoding_batch["facility_zip"]
    .str.extract(r"(\d{5})", expand=False)
)

review_geocoding_batch = review_geocoding_batch[
    [
        "facility_id",
        "original_address",
        "facility_address",
        "facility_city",
        "facility_state",
        "facility_zip"
    ]
]

print("Review rows prepared:", len(review_geocoding_batch))
print(
    "Duplicate facility IDs:",
    review_geocoding_batch["facility_id"].duplicated().sum()
)

missing_summary = (
    review_geocoding_batch[
        [
            "facility_address",
            "facility_city",
            "facility_state",
            "facility_zip"
        ]
    ]
    .isna()
    .sum()
    .rename_axis("column")
    .reset_index(name="missing_count")
)

display(missing_summary)
display(review_geocoding_batch.head(10))

Review rows prepared: 270
Duplicate facility IDs: 0


,column,missing_count
0,facility_address,0
1,facility_city,0
2,facility_state,0
3,facility_zip,0


,facility_id,original_address,facility_address,facility_city,facility_state,facility_zip
16,FA0013677,10855 LINDBROOK DR,10855 LINDBROOK DR,LOS ANGELES,CA,90024
21,FA0014798,6117 RESEDA BLVD STE B&C,6117 RESEDA BLVD,RESEDA,CA,91335
30,FA0016232,10990 LOWER AZUSA RD STE 2,10990 LOWER AZUSA RD,EL MONTE,CA,91731
36,FA0017693,704 W LAS TUNAS DR STE 5,704 W LAS TUNAS DR,SAN GABRIEL,CA,91776
39,FA0019766,460 S ROBERTSON BLVD,460 S ROBERTSON BLVD,LOS ANGELES,CA,90048
40,FA0020085,1733 COLORADO BLVD,1733 COLORADO BLVD,LOS ANGELES,CA,90041
41,FA0020314,1747 W REDONDO BEACH BLVD,1747 W REDONDO BEACH BLVD,GARDENA,CA,90247
44,FA0023057,8527 E ALONDRA BLVD STE 172,8527 E ALONDRA BLVD,PARAMOUNT,CA,90723
48,FA0026043,965 S GLENDORA AVE,965 S GLENDORA AVE,WEST COVINA,CA,91790
49,FA0026066,3403 W SLAUSON AVE,3403 W SLAUSON AVE,LOS ANGELES,CA,90043


In [6]:
geocoding_dir = (
    project_root
    / "data"
    / "interim"
    / "geocoding"
)

geocoding_dir.mkdir(
    parents=True,
    exist_ok=True
)

review_batch_path = (
    geocoding_dir
    / "review_candidates_geocoding_batch.csv"
)

census_batch = review_geocoding_batch[
    [
        "facility_id",
        "facility_address",
        "facility_city",
        "facility_state",
        "facility_zip"
    ]
].copy()

census_batch.to_csv(
    review_batch_path,
    index=False,
    header=False
)

print("Batch rows saved:", len(census_batch))
print("Batch file exists:", review_batch_path.exists())
print("Batch file path:", review_batch_path)

display(census_batch.head(10))

Batch rows saved: 270
Batch file exists: True
Batch file path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\review_candidates_geocoding_batch.csv


,facility_id,facility_address,facility_city,facility_state,facility_zip
16,FA0013677,10855 LINDBROOK DR,LOS ANGELES,CA,90024
21,FA0014798,6117 RESEDA BLVD,RESEDA,CA,91335
30,FA0016232,10990 LOWER AZUSA RD,EL MONTE,CA,91731
36,FA0017693,704 W LAS TUNAS DR,SAN GABRIEL,CA,91776
39,FA0019766,460 S ROBERTSON BLVD,LOS ANGELES,CA,90048
40,FA0020085,1733 COLORADO BLVD,LOS ANGELES,CA,90041
41,FA0020314,1747 W REDONDO BEACH BLVD,GARDENA,CA,90247
44,FA0023057,8527 E ALONDRA BLVD,PARAMOUNT,CA,90723
48,FA0026043,965 S GLENDORA AVE,WEST COVINA,CA,91790
49,FA0026066,3403 W SLAUSON AVE,LOS ANGELES,CA,90043


In [8]:
import requests

geocoder_url = (
    "https://geocoding.geo.census.gov"
    "/geocoder/locations/addressbatch"
)

review_result_path = (
    geocoding_dir
    / "review_candidates_geocoding_results.csv"
)

with open(review_batch_path, "rb") as batch_file:
    response = requests.post(
        geocoder_url,
        files={
            "addressFile": (
                review_batch_path.name,
                batch_file,
                "text/csv"
            )
        },
        data={
            "benchmark": "Public_AR_Current"
        },
        timeout=180
    )

print("Request status:", response.status_code)
print("Response type:", response.headers.get("Content-Type"))

response.raise_for_status()

review_result_path.write_bytes(response.content)

print("Result file exists:", review_result_path.exists())
print("Result file size:", review_result_path.stat().st_size, "bytes")
print("Result file path:", review_result_path)

Request status: 200
Response type: text/plain
Result file exists: True
Result file size: 44641 bytes
Result file path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\review_candidates_geocoding_results.csv


In [9]:
review_geocoded = pd.read_csv(
    review_result_path,
    header=None,
    dtype="string"
)

review_geocoded.columns = [
    "facility_id",
    "input_address",
    "match_status",
    "match_type",
    "matched_address",
    "matched_coordinates",
    "tiger_line_id",
    "side"
]

coordinates = review_geocoded[
    "matched_coordinates"
].str.split(
    ",",
    n=1,
    expand=True
)

review_geocoded["longitude"] = pd.to_numeric(
    coordinates[0],
    errors="coerce"
)

review_geocoded["latitude"] = pd.to_numeric(
    coordinates[1],
    errors="coerce"
)

status_summary = (
    review_geocoded["match_status"]
    .value_counts(dropna=False)
    .rename_axis("match_status")
    .reset_index(name="facility_count")
)

print("Output rows:", len(review_geocoded))
print(
    "Duplicate facility IDs:",
    review_geocoded["facility_id"].duplicated().sum()
)
print(
    "Missing longitude:",
    review_geocoded["longitude"].isna().sum()
)
print(
    "Missing latitude:",
    review_geocoded["latitude"].isna().sum()
)

display(status_summary)

display(
    review_geocoded[
        [
            "facility_id",
            "match_status",
            "match_type",
            "matched_address",
            "longitude",
            "latitude"
        ]
    ].head(10)
)

Output rows: 270
Duplicate facility IDs: 0
Missing longitude: 8
Missing latitude: 8


,match_status,facility_count
0,Match,262
1,No_Match,7
2,Tie,1


,facility_id,match_status,match_type,matched_address,longitude,latitude
0,FA0026043,Match,Exact,"965 S GLENDORA AVE, WEST COVINA, CA, 91790",-117.935379,34.058287
1,FA0052695,Match,Exact,"2050 SAWTELLE BLVD, LOS ANGELES, CA, 90025",-118.442991,34.040296
2,FA0248885,Match,Exact,"1136 S DIAMOND BAR BLVD, DIAMOND BAR, CA, 91765",-117.81061,34.001547
3,FA0355901,Match,Non_Exact,"2825 S DIAMOND BAR BLVD, DIAMOND BAR, CA, 91765",-117.838186,33.973729
4,FA0165259,Match,Exact,"3250 W OLYMPIC BLVD, LOS ANGELES, CA, 90006",-118.307486,34.052589
5,FA0354936,Match,Exact,"1212 S BALDWIN AVE, ARCADIA, CA, 91007",-118.054796,34.125909
6,FA0312973,Match,Exact,"1408 E VALLEY BLVD, ALHAMBRA, CA, 91801",-118.109754,34.079192
7,FA0399600,Match,Exact,"12009 WILSHIRE BLVD, LOS ANGELES, CA, 90025",-118.465614,34.04569
8,FA0268931,Match,Exact,"120 W FOOTHILL BLVD, MONROVIA, CA, 91016",-118.002337,34.15128
9,FA0017693,Match,Exact,"704 W LAS TUNAS DR, SAN GABRIEL, CA, 91776",-118.109853,34.102356


In [10]:
# Merge geocoding results back to the review-candidate attributes
review_candidates_geocoded = review_candidates.merge(
    review_geocoded[
        [
            "facility_id",
            "input_address",
            "match_status",
            "match_type",
            "matched_address",
            "longitude",
            "latitude"
        ]
    ],
    on="facility_id",
    how="left",
    validate="one_to_one"
)

# Keep records with usable coordinates
review_matched = review_candidates_geocoded.loc[
    review_candidates_geocoded["longitude"].notna()
    & review_candidates_geocoded["latitude"].notna()
].copy()

# Convert coordinates into spatial points
review_points = gpd.GeoDataFrame(
    review_matched,
    geometry=gpd.points_from_xy(
        review_matched["longitude"],
        review_matched["latitude"]
    ),
    crs="EPSG:4326"
)

# Load LA County Census tract boundaries
tract_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_demographics_2024.gpkg"
)

la_tracts = gpd.read_file(
    tract_path,
    layer="la_tract_demographics"
)

# Match the coordinate systems
review_points = review_points.to_crs(
    la_tracts.crs
)

# Check which Census tract contains each point
review_points_checked = gpd.sjoin(
    review_points,
    la_tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)

print("Review candidates after merge:", len(review_candidates_geocoded))
print("Geocoded review points:", len(review_points))
print(
    "Points inside an LA County tract:",
    review_points_checked["GEOID"].notna().sum()
)
print(
    "Points outside LA County tracts:",
    review_points_checked["GEOID"].isna().sum()
)
print(
    "Duplicate facility IDs after spatial join:",
    review_points_checked["facility_id"].duplicated().sum()
)

Review candidates after merge: 270
Geocoded review points: 262
Points inside an LA County tract: 262
Points outside LA County tracts: 0
Duplicate facility IDs after spatial join: 0


In [11]:
# Prepare the 262 spatially validated review-candidate points
review_points_valid = (
    review_points_checked.loc[
        review_points_checked["GEOID"].notna()
    ]
    .drop(columns=["index_right"], errors="ignore")
    .copy()
)

review_points_valid = review_points_valid.to_crs(
    "EPSG:4326"
)

review_points_valid["competition_scenario"] = (
    "expanded_review_candidate"
)

# Prepare the eight unresolved addresses
review_unresolved = review_candidates_geocoded.loc[
    review_candidates_geocoded["longitude"].isna()
    | review_candidates_geocoded["latitude"].isna()
].copy()

review_points_path = (
    project_root
    / "data"
    / "processed"
    / "review_candidates_geocoded.gpkg"
)

review_unresolved_path = (
    geocoding_dir
    / "review_candidates_unresolved.csv"
)

review_points_valid.to_file(
    review_points_path,
    layer="review_candidates_geocoded",
    driver="GPKG"
)

review_unresolved.to_csv(
    review_unresolved_path,
    index=False
)

print("Validated review points saved:", len(review_points_valid))
print("Unresolved review addresses saved:", len(review_unresolved))
print("GeoPackage exists:", review_points_path.exists())
print("Unresolved CSV exists:", review_unresolved_path.exists())
print("GeoPackage path:", review_points_path)
print("Unresolved CSV path:", review_unresolved_path)

Validated review points saved: 262
Unresolved review addresses saved: 8
GeoPackage exists: True
Unresolved CSV exists: True
GeoPackage path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\review_candidates_geocoded.gpkg
Unresolved CSV path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\interim\geocoding\review_candidates_unresolved.csv


In [13]:
# Load the previously accepted strong-candidate locations
strong_points_path = (
    project_root
    / "data"
    / "processed"
    / "chinese_restaurant_candidates_geocoded.gpkg"
)

strong_points_all = gpd.read_file(
    strong_points_path
)

accepted_quality = [
    "accepted_exact",
    "accepted_standardization"
]

strong_points_accepted = strong_points_all.loc[
    strong_points_all["geocode_quality"].isin(
        accepted_quality
    )
].copy()

strong_points_accepted["competition_source"] = (
    "strong_candidate"
)

strong_points_accepted["competition_source"] = (
    "strong_candidate"
)

# Convert review points to exactly the same CRS as strong points
review_points_aligned = review_points_valid.to_crs(
    strong_points_accepted.crs
).copy()

review_points_aligned["competition_source"] = (
    "review_candidate"
)

print("Strong-point CRS:", strong_points_accepted.crs)
print("Review-point CRS:", review_points_aligned.crs)

competition_columns = [
    "facility_id",
    "facility_name",
    "competition_source",
    "geometry"
]

combined_table = pd.concat(
    [
        strong_points_accepted[competition_columns],
        review_points_aligned[competition_columns]
    ],
    ignore_index=True
)

expanded_competitors = gpd.GeoDataFrame(
    combined_table,
    geometry="geometry",
    crs=strong_points_accepted.crs
)

expanded_competitor_path = (
    project_root
    / "data"
    / "processed"
    / "expanded_chinese_competitor_candidates.gpkg"
)

expanded_competitors.to_file(
    expanded_competitor_path,
    layer="expanded_chinese_competitors",
    driver="GPKG"
)

source_summary = (
    expanded_competitors["competition_source"]
    .value_counts()
    .rename_axis("competition_source")
    .reset_index(name="facility_count")
)

print("Accepted strong candidates:", len(strong_points_accepted))
print("Validated review candidates:", len(review_points_valid))
print("Expanded competitor points:", len(expanded_competitors))
print(
    "Duplicate facility IDs:",
    expanded_competitors["facility_id"].duplicated().sum()
)
print(
    "Missing geometry:",
    expanded_competitors.geometry.isna().sum()
)
print("Expanded GeoPackage exists:", expanded_competitor_path.exists())

display(source_summary)

Strong-point CRS: EPSG:4269
Review-point CRS: EPSG:4269
Accepted strong candidates: 510
Validated review candidates: 262
Expanded competitor points: 772
Duplicate facility IDs: 0
Missing geometry: 0
Expanded GeoPackage exists: True


,competition_source,facility_count
0,strong_candidate,510
1,review_candidate,262


In [14]:
# Load the tract-level site-selection dataset
tract_inputs_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_site_selection_inputs.gpkg"
)

tract_inputs = gpd.read_file(
    tract_inputs_path
)

# California Albers projection for distance measurement
distance_crs = "EPSG:3310"

tracts_projected = tract_inputs.to_crs(
    distance_crs
)

expanded_points_projected = expanded_competitors.to_crs(
    distance_crs
)

# Use tract centroids as the screening locations
tract_centroids = gpd.GeoDataFrame(
    tracts_projected[["GEOID"]].copy(),
    geometry=tracts_projected.geometry.centroid,
    crs=distance_crs
)


def count_competitors_within_radius(
    centroids,
    competitor_points,
    radius_meters,
    output_column
):
    buffers = gpd.GeoDataFrame(
        centroids[["GEOID"]].copy(),
        geometry=centroids.geometry.buffer(radius_meters),
        crs=centroids.crs
    )

    joined = gpd.sjoin(
        buffers,
        competitor_points[
            ["facility_id", "geometry"]
        ],
        how="left",
        predicate="contains"
    )

    counts = (
        joined.groupby("GEOID")["facility_id"]
        .count()
        .rename(output_column)
        .reset_index()
    )

    return counts


one_mile_meters = 1609.344
three_miles_meters = 4828.032

expanded_1_mile = count_competitors_within_radius(
    tract_centroids,
    expanded_points_projected,
    one_mile_meters,
    "expanded_competitors_within_1_mile"
)

expanded_3_miles = count_competitors_within_radius(
    tract_centroids,
    expanded_points_projected,
    three_miles_meters,
    "expanded_competitors_within_3_miles"
)

expanded_competition_metrics = (
    tract_inputs
    .merge(
        expanded_1_mile,
        on="GEOID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        expanded_3_miles,
        on="GEOID",
        how="left",
        validate="one_to_one"
    )
)

count_columns = [
    "expanded_competitors_within_1_mile",
    "expanded_competitors_within_3_miles"
]

expanded_competition_metrics[count_columns] = (
    expanded_competition_metrics[count_columns]
    .fillna(0)
    .astype("int64")
)

print("Tracts calculated:", len(expanded_competition_metrics))
print(
    "Maximum expanded competitors within 1 mile:",
    expanded_competition_metrics[
        "expanded_competitors_within_1_mile"
    ].max()
)
print(
    "Maximum expanded competitors within 3 miles:",
    expanded_competition_metrics[
        "expanded_competitors_within_3_miles"
    ].max()
)

display(
    expanded_competition_metrics[
        [
            "GEOID",
            "competitors_within_1_mile",
            "expanded_competitors_within_1_mile",
            "competitors_within_3_miles",
            "expanded_competitors_within_3_miles"
        ]
    ]
    .sort_values(
        "expanded_competitors_within_3_miles",
        ascending=False
    )
    .head(10)
)

Tracts calculated: 2498
Maximum expanded competitors within 1 mile: 29
Maximum expanded competitors within 3 miles: 95


,GEOID,competitors_within_1_mile,expanded_competitors_within_1_mile,competitors_within_3_miles,expanded_competitors_within_3_miles
2385,06037481712,8,19,51,95
1745,06037481401,10,17,47,94
672,06037481403,13,23,46,94
674,06037481404,12,23,46,93
1192,06037482304,3,6,48,93
1180,06037482301,12,20,47,92
2371,06037481002,6,11,46,92
2374,06037481103,8,18,44,91
1181,06037482303,13,22,46,91
2372,06037481101,3,9,44,91


In [15]:
expanded_competition_metrics[
    "additional_competitors_within_1_mile"
] = (
    expanded_competition_metrics[
        "expanded_competitors_within_1_mile"
    ]
    - expanded_competition_metrics[
        "competitors_within_1_mile"
    ]
)

expanded_competition_metrics[
    "additional_competitors_within_3_miles"
] = (
    expanded_competition_metrics[
        "expanded_competitors_within_3_miles"
    ]
    - expanded_competition_metrics[
        "competitors_within_3_miles"
    ]
)

print(
    "Negative 1-mile differences:",
    (
        expanded_competition_metrics[
            "additional_competitors_within_1_mile"
        ] < 0
    ).sum()
)

print(
    "Negative 3-mile differences:",
    (
        expanded_competition_metrics[
            "additional_competitors_within_3_miles"
        ] < 0
    ).sum()
)

comparison_columns = [
    "competitors_within_1_mile",
    "expanded_competitors_within_1_mile",
    "additional_competitors_within_1_mile",
    "competitors_within_3_miles",
    "expanded_competitors_within_3_miles",
    "additional_competitors_within_3_miles"
]

display(
    expanded_competition_metrics[
        comparison_columns
    ].describe().round(2)
)

sensitivity_csv_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_competition_sensitivity_metrics.csv"
)

sensitivity_gpkg_path = (
    project_root
    / "data"
    / "processed"
    / "la_tract_competition_sensitivity_metrics.gpkg"
)

expanded_competition_metrics.drop(
    columns="geometry"
).to_csv(
    sensitivity_csv_path,
    index=False
)

expanded_competition_metrics.to_file(
    sensitivity_gpkg_path,
    layer="competition_sensitivity_metrics",
    driver="GPKG"
)

print("CSV exists:", sensitivity_csv_path.exists())
print("GeoPackage exists:", sensitivity_gpkg_path.exists())
print("CSV path:", sensitivity_csv_path)
print("GeoPackage path:", sensitivity_gpkg_path)

Negative 1-mile differences: 0
Negative 3-mile differences: 0


,competitors_within_1_mile,expanded_competitors_within_1_mile,additional_competitors_within_1_mile,competitors_within_3_miles,expanded_competitors_within_3_miles,additional_competitors_within_3_miles
count,2498.00,2498.00,2498.00,2498.00,2498.00,2498.00
mean,1.93,2.84,0.91,14.01,20.57,6.56
std,2.41,3.83,1.83,11.84,17.98,7.59
min,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,0.00,0.00,6.00,9.00,2.00
50%,1.00,2.00,0.00,11.00,16.00,4.00
75%,3.00,4.00,1.00,18.00,25.00,8.00
max,16.00,29.00,16.00,52.00,95.00,48.00


CSV exists: True
GeoPackage exists: True
CSV path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_tract_competition_sensitivity_metrics.csv
GeoPackage path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_tract_competition_sensitivity_metrics.gpkg


In [16]:
eligible_reliability = [
    "more_reliable",
    "usable_with_caution"
]

sensitivity_scores = expanded_competition_metrics.loc[
    expanded_competition_metrics[
        "chinese_estimate_reliability"
    ].isin(eligible_reliability)
    & (expanded_competition_metrics["total_population"] > 0)
    & expanded_competition_metrics["chinese_share_pct"].notna()
    & expanded_competition_metrics[
        "median_household_income"
    ].notna()
].copy()

# Demand and income components remain unchanged
sensitivity_scores["chinese_population_rank"] = (
    sensitivity_scores["chinese_total_estimate"]
    .rank(pct=True, method="average")
)

sensitivity_scores["chinese_share_rank"] = (
    sensitivity_scores["chinese_share_pct"]
    .rank(pct=True, method="average")
)

sensitivity_scores["income_rank"] = (
    sensitivity_scores["median_household_income"]
    .rank(pct=True, method="average")
)

sensitivity_scores["demand_component"] = (
    0.60 * sensitivity_scores["chinese_population_rank"]
    + 0.40 * sensitivity_scores["chinese_share_rank"]
)

# Competition component under the original 510-candidate definition
sensitivity_scores["baseline_competition_component"] = (
    0.40
    * (
        1
        - sensitivity_scores["competitors_within_1_mile"]
        .rank(pct=True, method="average")
    )
    + 0.60
    * (
        1
        - sensitivity_scores["competitors_within_3_miles"]
        .rank(pct=True, method="average")
    )
)

# Competition component under the expanded 772-candidate definition
sensitivity_scores["expanded_competition_component"] = (
    0.40
    * (
        1
        - sensitivity_scores[
            "expanded_competitors_within_1_mile"
        ].rank(pct=True, method="average")
    )
    + 0.60
    * (
        1
        - sensitivity_scores[
            "expanded_competitors_within_3_miles"
        ].rank(pct=True, method="average")
    )
)

# Balanced score: 60% demand, 20% income, 20% competition
sensitivity_scores["baseline_score"] = (
    100
    * (
        0.60 * sensitivity_scores["demand_component"]
        + 0.20 * sensitivity_scores["income_rank"]
        + 0.20
        * sensitivity_scores["baseline_competition_component"]
    )
).round(2)

sensitivity_scores["expanded_score"] = (
    100
    * (
        0.60 * sensitivity_scores["demand_component"]
        + 0.20 * sensitivity_scores["income_rank"]
        + 0.20
        * sensitivity_scores["expanded_competition_component"]
    )
).round(2)

sensitivity_scores["baseline_rank"] = (
    sensitivity_scores["baseline_score"]
    .rank(ascending=False, method="min")
    .astype("int64")
)

sensitivity_scores["expanded_rank"] = (
    sensitivity_scores["expanded_score"]
    .rank(ascending=False, method="min")
    .astype("int64")
)

# Positive means the tract moved down under expanded competition
sensitivity_scores["rank_change"] = (
    sensitivity_scores["expanded_rank"]
    - sensitivity_scores["baseline_rank"]
)

baseline_top_20 = set(
    sensitivity_scores.loc[
        sensitivity_scores["baseline_rank"] <= 20,
        "GEOID"
    ]
)

expanded_top_20 = set(
    sensitivity_scores.loc[
        sensitivity_scores["expanded_rank"] <= 20,
        "GEOID"
    ]
)

top_20_overlap = baseline_top_20 & expanded_top_20

print("Eligible tracts:", len(sensitivity_scores))
print("Baseline top-20 tracts:", len(baseline_top_20))
print("Expanded top-20 tracts:", len(expanded_top_20))
print("Tracts remaining in both top 20:", len(top_20_overlap))
print(
    "Top-20 retention rate:",
    round(len(top_20_overlap) / 20 * 100, 2),
    "%"
)

display(
    sensitivity_scores[
        [
            "GEOID",
            "chinese_total_estimate",
            "chinese_share_pct",
            "median_household_income",
            "competitors_within_3_miles",
            "expanded_competitors_within_3_miles",
            "baseline_score",
            "expanded_score",
            "baseline_rank",
            "expanded_rank",
            "rank_change"
        ]
    ]
    .sort_values("expanded_rank")
    .head(20)
)

Eligible tracts: 964
Baseline top-20 tracts: 20
Expanded top-20 tracts: 20
Tracts remaining in both top 20: 17
Top-20 retention rate: 85.0 %


,GEOID,chinese_total_estimate,chinese_share_pct,median_household_income,competitors_within_3_miles,expanded_competitors_within_3_miles,baseline_score,expanded_score,baseline_rank,expanded_rank,rank_change
1679,06037403407,1661,70.14,156552.0,6,10,91.01,90.90,1,1,0
1657,06037403404,1035,48.96,173289.0,7,10,88.61,89.15,5,2,-3
141,06037408503,1943,29.42,145921.0,7,8,87.62,88.87,8,3,-5
1870,06037670413,881,17.99,214625.0,0,0,88.16,88.45,7,4,-3
239,06037403325,2735,58.12,111445.0,2,7,89.06,88.26,4,5,1
1671,06037430400,1674,39.66,182292.0,8,20,89.53,87.03,2,6,4
699,06037464102,2617,57.62,239052.0,16,39,89.11,86.28,3,7,4
2055,06037670326,680,21.36,250001.0,3,7,86.66,86.24,13,8,-5
1023,06037403324,2944,45.87,114547.0,6,13,87.21,85.97,10,9,-1
1503,06037403403,2654,52.44,140795.0,10,16,84.99,85.89,15,10,-5


In [17]:
sensitivity_scores["in_baseline_top_20"] = (
    sensitivity_scores["baseline_rank"] <= 20
)

sensitivity_scores["in_expanded_top_20"] = (
    sensitivity_scores["expanded_rank"] <= 20
)

sensitivity_scores["competition_robustness"] = np.select(
    [
        (
            sensitivity_scores["in_baseline_top_20"]
            & sensitivity_scores["in_expanded_top_20"]
        ),
        (
            sensitivity_scores["in_baseline_top_20"]
            & ~sensitivity_scores["in_expanded_top_20"]
        ),
        (
            ~sensitivity_scores["in_baseline_top_20"]
            & sensitivity_scores["in_expanded_top_20"]
        )
    ],
    [
        "stable_top_20",
        "drops_from_top_20",
        "enters_top_20"
    ],
    default="outside_top_20"
)

robustness_summary = (
    sensitivity_scores["competition_robustness"]
    .value_counts()
    .rename_axis("competition_robustness")
    .reset_index(name="tract_count")
)

display(robustness_summary)

changed_top_20 = sensitivity_scores.loc[
    sensitivity_scores["competition_robustness"].isin(
        [
            "drops_from_top_20",
            "enters_top_20"
        ]
    ),
    [
        "GEOID",
        "competitors_within_3_miles",
        "expanded_competitors_within_3_miles",
        "baseline_score",
        "expanded_score",
        "baseline_rank",
        "expanded_rank",
        "rank_change",
        "competition_robustness"
    ]
].sort_values(
    ["competition_robustness", "expanded_rank"]
)

display(changed_top_20)

sensitivity_score_csv_path = (
    project_root
    / "data"
    / "processed"
    / "la_competition_definition_sensitivity_scores.csv"
)

sensitivity_score_gpkg_path = (
    project_root
    / "data"
    / "processed"
    / "la_competition_definition_sensitivity.gpkg"
)

sensitivity_scores.drop(
    columns="geometry"
).to_csv(
    sensitivity_score_csv_path,
    index=False
)

sensitivity_scores.to_file(
    sensitivity_score_gpkg_path,
    layer="competition_definition_sensitivity",
    driver="GPKG"
)

print("Sensitivity rows saved:", len(sensitivity_scores))
print("CSV exists:", sensitivity_score_csv_path.exists())
print("GeoPackage exists:", sensitivity_score_gpkg_path.exists())
print("CSV path:", sensitivity_score_csv_path)
print("GeoPackage path:", sensitivity_score_gpkg_path)

,competition_robustness,tract_count
0,outside_top_20,941
1,stable_top_20,17
2,drops_from_top_20,3
3,enters_top_20,3


,GEOID,competitors_within_3_miles,expanded_competitors_within_3_miles,baseline_score,expanded_score,baseline_rank,expanded_rank,rank_change,competition_robustness
497,06037460002,3,9,83.52,82.43,19,24,5,drops_from_top_20
1712,06037403408,12,32,84.43,81.82,17,28,11,drops_from_top_20
71,06037408707,14,34,86.22,80.40,14,35,21,drops_from_top_20
596,06037403409,7,8,81.96,83.70,30,17,-13,enters_top_20
2057,06037670416,0,0,83.04,83.33,21,19,-2,enters_top_20
121,06037670702,3,4,82.63,83.03,22,20,-2,enters_top_20


Sensitivity rows saved: 964
CSV exists: True
GeoPackage exists: True
CSV path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_competition_definition_sensitivity_scores.csv
GeoPackage path: c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_competition_definition_sensitivity.gpkg
